In [1]:
import Util.math_functions as mathf

import numpy as np

from Util.Problems import Problem, solution


class P018(Problem):
    number = 18
    title = "Maximum Path Sum I"
    description = """<p>By starting at the top of the triangle below and moving to adjacent numbers on the row below, the maximum total from top to bottom is $23$.</p><p class="monospace center"><span class="red"><b>3</b></span><br/><span class="red"><b>7</b></span> 4<br/>
2 <span class="red"><b>4</b></span> 6<br/>
8 5 <span class="red"><b>9</b></span> 3</p><p>That is, $3 + 7 + 4 + 9 = 23$.</p><p>Find the maximum total from top to bottom of the triangle below:</p><p class="monospace center narrow_device_shrink copy_to_clipboard">75<br/>
95 64<br/>
17 47 82<br/>
18 35 87 10<br/>
20 04 82 47 65<br/>
19 01 23 75 03 34<br/>
88 02 77 73 07 63 67<br/>
99 65 04 28 06 16 70 92<br/>
41 41 26 56 83 40 80 70 33<br/>
41 48 72 33 47 32 37 16 94 29<br/>
53 71 44 65 25 43 91 52 97 51 14<br/>
70 11 33 28 77 73 17 78 39 68 17 57<br/>
91 71 52 38 17 14 91 43 58 50 27 29 48<br/>
63 66 04 68 89 53 67 30 73 16 69 87 40 31<br/>
04 62 98 27 23 09 70 98 73 93 38 53 60 04 23</p><p class="smallest"><b>NOTE:</b> As there are only $16384$ routes, it is possible to solve this problem by trying every route. However, <a href="problem=67">Problem 67</a>, is the same challenge with a triangle containing one-hundred rows; it cannot be solved by brute force, and requires a clever method! (c;</p>"""
    triangle = """75
95 64
17 47 82
18 35 87 10
20 04 82 47 65
19 01 23 75 03 34
88 02 77 73 07 63 67
99 65 04 28 06 16 70 92
41 41 26 56 83 40 80 70 33
41 48 72 33 47 32 37 16 94 29
53 71 44 65 25 43 91 52 97 51 14
70 11 33 28 77 73 17 78 39 68 17 57
91 71 52 38 17 14 91 43 58 50 27 29 48
63 66 04 68 89 53 67 30 73 16 69 87 40 31
04 62 98 27 23 09 70 98 73 93 38 53 60 04 23"""

In [2]:
p = P018()
p.describe()

## Problem 18: Maximum Path Sum I

<p>By starting at the top of the triangle below and moving to adjacent numbers on the row below, the maximum total from top to bottom is $23$.</p><p class="monospace center"><span class="red"><b>3</b></span><br/><span class="red"><b>7</b></span> 4<br/>
2 <span class="red"><b>4</b></span> 6<br/>
8 5 <span class="red"><b>9</b></span> 3</p><p>That is, $3 + 7 + 4 + 9 = 23$.</p><p>Find the maximum total from top to bottom of the triangle below:</p><p class="monospace center narrow_device_shrink copy_to_clipboard">75<br/>
95 64<br/>
17 47 82<br/>
18 35 87 10<br/>
20 04 82 47 65<br/>
19 01 23 75 03 34<br/>
88 02 77 73 07 63 67<br/>
99 65 04 28 06 16 70 92<br/>
41 41 26 56 83 40 80 70 33<br/>
41 48 72 33 47 32 37 16 94 29<br/>
53 71 44 65 25 43 91 52 97 51 14<br/>
70 11 33 28 77 73 17 78 39 68 17 57<br/>
91 71 52 38 17 14 91 43 58 50 27 29 48<br/>
63 66 04 68 89 53 67 30 73 16 69 87 40 31<br/>
04 62 98 27 23 09 70 98 73 93 38 53 60 04 23</p><p class="smallest"><b>NOTE:</b> As there are only $16384$ routes, it is possible to solve this problem by trying every route. However, <a href="problem=67">Problem 67</a>, is the same challenge with a triangle containing one-hundred rows; it cannot be solved by brute force, and requires a clever method! (c;</p>

### Solution notes
As mentioned in the problem, this is easily brute-forceable. I also agree with the problem statement that this will not be the final solution. However, I do want to try it first to see what kind of time we will be dealing with. This function does not look at any of the values. It makes a list of all possible paths in a triangle, then calculates the sums that results in for this particular triangle and finds the highest.

In [3]:
@solution(P018, first=True, max_tests=10, make_fast=False, warmup_args=(P018.triangle, ))
def brute_force_list(triangle):
    rows =  triangle.splitlines()
    triangle_size = len(rows[-1].split())
    triangle_array = np.zeros((len(rows), triangle_size), dtype=int)
    for i, row in enumerate(rows):
        row_items = row.split()
        triangle_array[i][:len(row_items)] = row_items
    paths = [[0]]
    for row in range(1, triangle_size):
        new_paths = []
        for path in paths:
            new_paths.append(path + [path[-1]])
            new_paths.append(path + [path[-1] + 1])
        paths = new_paths
    highest_score = 0
    for path in paths:
        path_score = 0
        for i, row in enumerate(triangle_array):
            path_score += row[path[i]]
        highest_score = max(path_score, highest_score)
    return highest_score

In [4]:
p.test_all()

1074 found after 10 tests in 79.535270 ms by brute_force_list (first)


Numba was being annoying, as per usual, so the first version was just plain slow Python. After some additional tinkering, I was able to get the same algorithm to run with Numba.

In [5]:
@solution(P018, max_tests= 100, make_fast=True, warmup_args=(P018.triangle, ))
def numba_brute_force_list(triangle):
    rows =  triangle.splitlines()
    triangle_size = len(rows[-1].split())
    triangle_array = np.zeros((len(rows), triangle_size), dtype=np.int_)
    for i, row in enumerate(rows):
        row_items = row.split()
        for j, item in enumerate(row_items):
            triangle_array[i][j] = mathf.string_to_int(item)
    paths = [[0]]
    for row in range(1, triangle_size):
        new_paths = []
        for path in paths:
            new_paths.append(path + [path[-1]])
            new_paths.append(path + [path[-1] + 1])
        paths = new_paths
    highest_score = 0
    for path in paths:
        path_score = 0
        for i, row in enumerate(triangle_array):
            path_score += row[path[i]]
        highest_score = max(path_score, highest_score)
    return highest_score

In [6]:
p.test_all()

1074 found after 10 tests in 39.830820 ms by brute_force_list (first)
1074 found after 100 tests in 5.484899 ms by numba_brute_force_list


With the bottom-up approach, we start by looking at the second-last row. When we reach any of these items, the last choice is trivial: we pick the highest of the two options in the last row. So in this algorithm, the highest of the two final options for the second-last row, are added to its value. This creates a new triangle, to which we can apply this same logic, looking at the new second-last row and adding the best final option to each value. We keep doing this until we reach the top, to find the score of the highest path.

In [7]:
@solution(P018, best=True, make_fast=True, warmup_args=(P018.triangle, ))
def bottom_up(triangle):
    rows =  triangle.splitlines()
    triangle_size = len(rows[-1].split())
    triangle_array = np.zeros((len(rows), triangle_size), dtype=np.int_)
    for i, row in enumerate(rows):
        row_items = row.split()
        for j, item in enumerate(row_items):
            triangle_array[i][j] = mathf.string_to_int(item)
    for i in range(triangle_size - 1, 0, -1):
        for index in range(i):
            # print(i, index)
            second_last_row_item = triangle_array[i - 1][index]
            # print(second_last_row_item)
            second_last_row_item += max(triangle_array[i][index], triangle_array[i][index + 1])
            triangle_array[i - 1][index] = second_last_row_item
    return triangle_array[0][0]

In [8]:
p.test_all()

1074 found after 1000 tests in 0.016842 ms by bottom_up (best)
1074 found after 10 tests in 36.906480 ms by brute_force_list (first)
1074 found after 100 tests in 5.280837 ms by numba_brute_force_list
